# Σημειώσεις Pandas: Αριστερή Σύνδεση Πινάκων (`Left Merge`)

Το **Left Merge** (`how="left"`) είναι το πιο συχνά χρησιμοποιούμενο join στην ανάλυση δεδομένων και συμπεριφοράς (Behavioral Analytics). 

## 💡 Η Βασική Λογική
* **Κρατάει ΤΑ ΠΑΝΤΑ** από τον αριστερό πίνακα - τον πληθυσμό (π.χ. όλους τους πελάτες ή χρήστες). **Καμία γραμμή του δεν διαγράφεται.**
* Φέρνει τις αντίστοιχες πληροφορίες από τον δεξιό πίνακα (π.χ. τις παραγγελίες τους) όπου υπάρχει ταίρι.
* Όταν μια εγγραφή του αριστερού πίνακα **δεν** έχει ταίρι στον δεξιό, το Pandas συμπληρώνει τις στήλες του δεξιού πίνακα με **`NaN`**.

---

## 🔍 Βασική Σύνταξη

```python
import pandas as pd

# Κρατάμε όλους τους πελάτες (αριστερά) και φέρνουμε τις παραγγελίες τους (δεξιά)
all_customers = pd.merge(
    customers,          # Αριστερός πίνακας (πρωταγωνιστής)
    orders,             # Δεξιός πίνακας (επιπλέον πληροφορίες)
    how="left",          # Ορίζει το Left Merge
    left_on="id",       # Κλειδί αριστερού πίνακα
    right_on="cust_id"  # Κλειδί δεξιού πίνακα 
)
```

---

## 🎯 Κλασικό Μοτίβο Συνέντευξης: Εύρεση «Απουσιών» (Anti-Join)

Πολλές ερωτήσεις συμπεριφοράς αφορούν **όσους ΔΕΝ έκαναν μια ενέργεια** (π.χ. *«Ποιοι πελάτες δεν αγόρασαν ποτέ;»*, *«Ποια προϊόντα δεν πουλήθηκαν;»*, *«Ποιοι χρήστες δεν έκαναν login;»*).

### Βήμα 1: Κάνουμε `how="left"`
Βάζουμε τον συνολικό πληθυσμό στα αριστερά και τις ενέργειες στα δεξιά.

### Βήμα 2: Φιλτράρουμε τα `NaN` με `.isna()`
Όσοι δεν έκαναν την ενέργεια θα έχουν `NaN` στις στήλες του δεξιού πίνακα.

```python
# Βρίσκουμε τους πελάτες που δεν έχουν καμία παραγγελία
inactive_customers = all_customers[all_customers["order_id"].isna()]
```

---

## 💡 Τι γίνεται με το Right Merge (`how="right"`);
* Κάνει ακριβώς το αντίστροφο (κρατάει τα πάντα από τον δεξιό πίνακα).
* **Στην πράξη δεν χρησιμοποιείται σχεδόν ποτέ.** Αντί να γράψουμε `pd.merge(A, B, how="right")`, απλώς αλλάζουμε τη σειρά των πινάκων και γράφουμε `pd.merge(B, A, how="left")` για να είναι πιο ευανάγνωστος ο κώδικας.


In [1]:
import pandas as pd
import numpy as np

# 1. Πίνακας Πελατών (Primary Key: 'id')
# Σημείωση: Ο πελάτης 104 (Δημήτρης) δεν έχει κάνει καμία παραγγελία!
customers = pd.DataFrame({
    'id':[101, 102, 103, 104, 105],
    'name': ['Maria', 'Nikos', 'Eleni', 'Dimitris', 'Panos'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze', np.nan]
})

# 2. Πίνακας Παραγγελιών (Foreign Key: 'cust_id')
# Σημείωση: Η παραγγελία ORD-5 ανήκει στον πελάτη 999 που δεν υπάρχει στον πίνακα customers!
orders = pd.DataFrame({
    'order_id': ['ORD-1', 'ORD-2', 'ORD-3', 'ORD-4', 'ORD-5'],
    'cust_id':[101, 101, 103, 102, 999],
    'amount': [120.50, 45.00, 89.90, 210.00, 30.00]
})


In [2]:
#Left merge customers with orders so every customer appears, even those with no orders.
all_customers = pd.merge(customers, orders, how='left', left_on='id' ,right_on='cust_id')
all_customers

,id,name,tier,order_id,cust_id,amount
0,101,Maria,Gold,ORD-1,101.0,120.5
1,101,Maria,Gold,ORD-2,101.0,45.0
2,102,Nikos,Silver,ORD-4,102.0,210.0
3,103,Eleni,Gold,ORD-3,103.0,89.9
4,104,Dimitris,Bronze,NaN,NaN,NaN
5,105,Panos,NaN,NaN,NaN,NaN


In [3]:
# Anti-Join, “Find customers who have never placed an order.”
never_ordered = all_customers[all_customers['order_id'].isna()]
never_ordered

,id,name,tier,order_id,cust_id,amount
4,104,Dimitris,Bronze,NaN,NaN,NaN
5,105,Panos,NaN,NaN,NaN,NaN


In [4]:
# βάλε ετικέτα σε αυτούς που δεν έχουν παραγγείλει ποτέ και δεν έχουν tier, 
# ως 'inactive', ενώ στους υπόλοιπους 'active'
all_customers['label'] = np.where(
    (all_customers['order_id'].isna())&
    (all_customers['tier'].isna()),
    'inactive', 'active'
)
all_customers

,id,name,tier,order_id,cust_id,amount,label
0,101,Maria,Gold,ORD-1,101.0,120.5,active
1,101,Maria,Gold,ORD-2,101.0,45.0,active
2,102,Nikos,Silver,ORD-4,102.0,210.0,active
3,103,Eleni,Gold,ORD-3,103.0,89.9,active
4,104,Dimitris,Bronze,NaN,NaN,NaN,active
5,105,Panos,NaN,NaN,NaN,NaN,inactive


# Σημειώσεις Pandas: Έλεγχος Σύνδεσης με `indicator=True`

Η παράμετρος `indicator=True` στη συνάρτηση `pd.merge()` είναι ένα εργαλείο ελέγχου (debugging) που προσθέτει αυτόματα μια βοηθητική στήλη με όνομα **`_merge`** στο τέλος του πίνακα, δείχνοντας την προέλευση κάθε γραμμής.

---

## 🔍 Τιμές της στήλης `_merge`

* **`both`**: Το κλειδί βρέθηκε **και στους δύο πίνακες** (επιτυχές ταίριασμα).
* **`left_only`**: Η εγγραφή υπήρχε **μόνο στον αριστερό πίνακα** (π.χ. πελάτης χωρίς παραγγελίες).
* **`right_only`**: Η εγγραφή υπήρχε **μόνο στον δεξιό πίνακα** *(εμφανίζεται μόνο σε `right` ή `outer` merge)*.

---

## 💻 Παράδειγμα Χρήσης

```python
import pandas as pd

result = pd.merge(
    customers, 
    orders, 
    how="left",
    left_on="id", 
    right_on="cust_id", 
    indicator=True
)

# 1. Γρήγορη σύνοψη του πόσες εγγραφές ταίριαξαν
print(result["_merge"].value_counts())

# 2. Εύρεση πελατών χωρίς παραγγελίες (πιο καθαρό από το .isna())
no_orders = result[result["_merge"] == "left_only"]

# 3. Διαγραφή της βοηθητικής στήλης αφού τελειώσουμε τον έλεγχο
result = result.drop(columns=["_merge"])
```

---

## 💡 Γιατί να το χρησιμοποιήσω;
* **Ταχύτητα ελέγχου:** Με ένα απλό `.value_counts()` βλέπεις αμέσως αν έχασες δεδομένα ή αν υπάρχουν ορφανές εγγραφές, χωρίς να ψάχνεις για `NaN` σε πολλές στήλες.
* **Ευανάγνωστο φιλτράρισμα:** Το `result["_merge"] == "left_only"` περιγράφει με λόγια αυτό που ψάχνεις, κάνοντας τον κώδικα πιο κατανοητό.


In [5]:
l_merged= pd.merge(customers, orders, how='left', left_on='id', right_on='cust_id', indicator=True)
l_merged

,id,name,tier,order_id,cust_id,amount,_merge
0,101,Maria,Gold,ORD-1,101.0,120.5,both
1,101,Maria,Gold,ORD-2,101.0,45.0,both
2,102,Nikos,Silver,ORD-4,102.0,210.0,both
3,103,Eleni,Gold,ORD-3,103.0,89.9,both
4,104,Dimitris,Bronze,NaN,NaN,NaN,left_only
5,105,Panos,NaN,NaN,NaN,NaN,left_only


In [6]:
l_merged['_merge'].value_counts()

_merge
both          4
left_only     2
right_only    0
Name: count, dtype: int64

In [7]:
# Anti-Join, “Find customers who have never placed an order.”
no_orders = l_merged[l_merged['_merge']=='left_only']
no_orders

,id,name,tier,order_id,cust_id,amount,_merge
4,104,Dimitris,Bronze,NaN,NaN,NaN,left_only
5,105,Panos,NaN,NaN,NaN,NaN,left_only


In [8]:
l_merged = l_merged.drop(columns='_merge')
l_merged

,id,name,tier,order_id,cust_id,amount
0,101,Maria,Gold,ORD-1,101.0,120.5
1,101,Maria,Gold,ORD-2,101.0,45.0
2,102,Nikos,Silver,ORD-4,102.0,210.0
3,103,Eleni,Gold,ORD-3,103.0,89.9
4,104,Dimitris,Bronze,NaN,NaN,NaN
5,105,Panos,NaN,NaN,NaN,NaN


# Σημειώσεις Pandas: Πλήρης Σύνδεση Πινάκων (`Outer Merge`)

Το **Outer Merge** (`how="outer"`) διατηρεί **όλες τις εγγραφές και από τους δύο πίνακες**, ανεξάρτητα από το αν ταιριάζουν μεταξύ τους. Όπου δεν υπάρχει αντιστοιχία, συμπληρώνει με **`NaN`**.

---

## 💻 Βασική Σύνταξη & Το «Όπλο» `indicator=True`

Ο συνδυασμός του `how="outer"` με το `indicator=True` είναι ο πιο γρήγορος τρόπος για **Gap Analysis** (να βρεις τι λείπει και από τις δύο πλευρές).

```python
import pandas as pd

result = pd.merge(
    customers, 
    orders, 
    how="outer", 
    left_on="id", 
    right_on="cust_id", 
    indicator=True
)

# Η στήλη '_merge' δείχνει ακριβώς τι συνέβη σε κάθε γραμμή:
# - 'both'       -> Υπάρχει και στους δύο πίνακες
# - 'left_only'  -> Πελάτης χωρίς καμία παραγγελία
# - 'right_only' -> Παραγγελία χωρίς έγκυρο πελάτη (ορφανή εγγραφή)
```

---

## 🎯 Κλασικό Μοτίβο Συνέντευξης («Βρες όλες τις ασυμφωνίες»)

**Ερώτηση:** *«Βρες όλους τους πελάτες που δεν έχουν παραγγελίες ΚΑΙ όλες τις παραγγελίες που δεν ανήκουν σε κάποιον πελάτη.»*

```python
# Φιλτράρουμε κρατώντας ό,τι ΔΕΝ ταίριαξε (δηλαδή != 'both')
unmatched = result[result["_merge"] != "both"]
```


In [9]:
combined= pd.merge(customers, orders, how='outer', left_on='id', right_on='cust_id', indicator=True)

In [10]:
combined

,id,name,tier,order_id,cust_id,amount,_merge
0,101.0,Maria,Gold,ORD-1,101.0,120.5,both
1,101.0,Maria,Gold,ORD-2,101.0,45.0,both
2,102.0,Nikos,Silver,ORD-4,102.0,210.0,both
3,103.0,Eleni,Gold,ORD-3,103.0,89.9,both
4,104.0,Dimitris,Bronze,NaN,NaN,NaN,left_only
5,105.0,Panos,NaN,NaN,NaN,NaN,left_only
6,NaN,NaN,NaN,ORD-5,999.0,30.0,right_only


In [11]:
gap_analysis = combined[combined['id'].isna() | combined['cust_id'].isna()]
gap_analysis

,id,name,tier,order_id,cust_id,amount,_merge
4,104.0,Dimitris,Bronze,NaN,NaN,NaN,left_only
5,105.0,Panos,NaN,NaN,NaN,NaN,left_only
6,NaN,NaN,NaN,ORD-5,999.0,30.0,right_only


In [12]:
# ή εφόσον έχω το _merge
gap_analysis = combined[combined['_merge']!='both']
gap_analysis

,id,name,tier,order_id,cust_id,amount,_merge
4,104.0,Dimitris,Bronze,NaN,NaN,NaN,left_only
5,105.0,Panos,NaN,NaN,NaN,NaN,left_only
6,NaN,NaN,NaN,ORD-5,999.0,30.0,right_only


In [13]:
gap_analysis['_merge'].value_counts()

_merge
left_only     2
right_only    1
both          0
Name: count, dtype: int64

# Σημειώσεις Pandas: Όλοι οι Δυνατοί Συνδυασμοί (`Cross Merge`)

Το **Cross Merge** (`how="cross"`) συνδυάζει **κάθε γραμμή του αριστερού πίνακα με κάθε γραμμή του δεξιού** (καρτεσιανό γινόμενο: N × M γραμμές).

---

## 💻 Βασική Σύνταξη

⚠️ **Σημαντική λεπτομέρεια:** Στο `how="cross"` **ΔΕΝ** βάζουμε `on=`, `left_on=` ή `right_on=`, γιατί δεν ψάχνουμε κοινό κλειδί· θέλουμε να ενωθούν τα πάντα με τα πάντα.

```python
import pandas as pd

sizes = pd.DataFrame({"size": ["S", "M", "L"]})
colors = pd.DataFrame({"color": ["Black", "White"]})

# Παράγει 3 x 2 = 6 συνδυασμούς
skus = pd.merge(sizes, colors, how="cross")
```

---

## 🎯 Κλασικό Μοτίβο Ανάλυσης: Date Scaffold (Μηδενικές Πωλήσεις)

Στα logs πωλήσεων, αν ένα προϊόν δεν πουλήθηκε τη Δευτέρα, δεν υπάρχει καν γραμμή για τη Δευτέρα. Για να εμφανίζονται και τα μηδενικά στα reports:

1. **`how="cross"`:** Ενώνουμε όλες τις ημερομηνίες (`dates`) με όλα τα προϊόντα (`products`) για να φτιάξουμε το πλήρες πλέγμα.
2. **`how="left"`:** Ενώνουμε το πλέγμα με τις πραγματικές πωλήσεις (`sales`) και γεμίζουμε τα `NaN` με `0` (`.fillna(0)`).


In [14]:
import pandas as pd
sizes = pd.DataFrame({"size": ["S", "M", "L"]})
colors = pd.DataFrame({"color": ["Black", "White"]})

# Παράγει 3 x 2 = 6 συνδυασμούς
skus = pd.merge(sizes, colors, how="cross")
skus

,size,color
0,S,Black
1,S,White
2,M,Black
3,M,White
4,L,Black
5,L,White
